# 02 -- Flask complet

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- creer une application Flask avec routes et vues typees ;
- gerer les methodes HTTP (GET, POST, PUT, DELETE) ;
- utiliser les templates Jinja2 ;
- gerer les formulaires et la validation ;
- construire une API REST JSON ;
- structurer l'application avec des blueprints ;
- gerer erreurs, logs, sessions et cookies ;
- tester l'application avec le client de test.

## Prerequis -- ce que vous devez maitriser

- fonctions typees et decorateurs ;
- HTTP : methodes, codes de statut, headers ;
- JSON.

Ce que ce notebook **ne couvrira pas** :

- FastAPI (voir `01_FastAPI_complet.ipynb`) ;
- SQLAlchemy en detail (voir `03_SQLAlchemy_avance.ipynb`) ;
- le deploiement production.

## Plan

1. Installation et Hello World
2. Routes et parametres
3. Methodes HTTP
4. Templates Jinja2
5. Formulaires
6. API REST JSON
7. Blueprints
8. Gestion des erreurs
9. Sessions et cookies
10. Middlewares et hooks
11. Configuration
12. Tests avec le client
13. Synthese
14. Exercices

---

## 1. Installation et Hello World

In [ ]:
# pip install flask
from flask import Flask, request, jsonify, render_template_string, session
import flask
flask.__version__

In [ ]:
app = Flask(__name__)

@app.route('/')
def index() -> str:
    return 'Hello Flask!'

print('app creee')

En production : `flask --app monapp run`. En notebook, on utilise le **client de test** pour invoquer les routes.

In [ ]:
with app.test_client() as client:
    resp = client.get('/')
    print(resp.data.decode())

---

## 2. Routes et parametres

In [ ]:
@app.route('/hello/<name>')
def hello(name: str) -> str:
    return f'Hello {name}!'

with app.test_client() as client:
    print(client.get('/hello/World').data.decode())

In [ ]:
@app.route('/user/<int:uid>')
def user_profile(uid: int) -> str:
    return f'Utilisateur #{uid}'

with app.test_client() as c:
    print(c.get('/user/42').data.decode())

### Convertisseurs

| Convertisseur | Description |
|---|---|
| `string` (defaut) | tout sauf `/` |
| `int` | entier positif |
| `float` | flottant positif |
| `path` | comme string mais accepte `/` |
| `uuid` | UUID |

---

## 3. Methodes HTTP

In [ ]:
@app.route('/items', methods=['GET', 'POST'])
def items():
    if request.method == 'POST':
        return jsonify({'created': request.get_json()}), 201
    return jsonify({'items': ['a', 'b', 'c']})

In [ ]:
with app.test_client() as c:
    print('GET :', c.get('/items').get_json())
    r = c.post('/items', json={'name': 'X'})
    print('POST:', r.get_json(), r.status_code)

In [ ]:
# Route dediee par methode
@app.get('/ping')
def ping():
    return {'pong': True}

with app.test_client() as c:
    print(c.get('/ping').get_json())

---

## 4. Templates Jinja2

In [ ]:
TEMPLATE = '''<html><body>
<h1>{{ title }}</h1>
<ul>
{% for item in items %}<li>{{ item }}</li>{% endfor %}
</ul>
</body></html>'''

@app.route('/page')
def page():
    return render_template_string(TEMPLATE, title='Ma page', items=['Python', 'Flask', 'Jinja2'])

with app.test_client() as c:
    print(c.get('/page').data.decode()[:120])

### Filtres Jinja2 courants

- `{{ nom|upper }}` / `|lower`
- `{{ valeur|default('N/A') }}`
- `{{ list|length }}`
- `{{ date|strftime('%d/%m/%Y') }}`

---

## 5. Formulaires

In [ ]:
@app.route('/form', methods=['GET', 'POST'])
def form():
    if request.method == 'POST':
        name = request.form.get('name', '')
        return f'Hello {name}'
    return '<form method=post><input name=name><button>OK</button></form>'

with app.test_client() as c:
    print(c.post('/form', data={'name': 'Alice'}).data.decode())

In [ ]:
# Validation simple
@app.post('/register')
def register():
    data = request.form
    if not data.get('email') or '@' not in data.get('email', ''):
        return {'error': 'email invalide'}, 400
    return {'ok': True, 'email': data['email']}

with app.test_client() as c:
    print(c.post('/register', data={'email': 'bad'}).status_code)
    print(c.post('/register', data={'email': 'a@b.c'}).get_json())

---

## 6. API REST JSON

In [ ]:
notes: dict[int, dict] = {}
next_id = [1]

@app.get('/api/notes')
def list_notes():
    return jsonify(list(notes.values()))

@app.post('/api/notes')
def create_note():
    data = request.get_json()
    note = {'id': next_id[0], 'title': data.get('title', ''), 'content': data.get('content', '')}
    notes[next_id[0]] = note
    next_id[0] += 1
    return jsonify(note), 201

@app.delete('/api/notes/<int:nid>')
def delete_note(nid: int):
    notes.pop(nid, None)
    return '', 204

In [ ]:
with app.test_client() as c:
    print(c.post('/api/notes', json={'title': 'Hi'}).get_json())
    print(c.get('/api/notes').get_json())
    print(c.delete('/api/notes/1').status_code)

---

## 7. Blueprints

Les **blueprints** decoupent une grosse app en modules reutilisables.

In [ ]:
from flask import Blueprint

admin_bp = Blueprint('admin', __name__, url_prefix='/admin')

@admin_bp.get('/dashboard')
def dashboard():
    return {'section': 'admin dashboard'}

app2 = Flask(__name__)
app2.register_blueprint(admin_bp)

with app2.test_client() as c:
    print(c.get('/admin/dashboard').get_json())

---

## 8. Gestion des erreurs

In [ ]:
@app.errorhandler(404)
def not_found(err):
    return jsonify({'error': 'not found'}), 404

@app.errorhandler(500)
def server_error(err):
    return jsonify({'error': 'internal'}), 500

with app.test_client() as c:
    print(c.get('/inexistant').status_code, c.get('/inexistant').get_json())

In [ ]:
# Lever une exception HTTP
from flask import abort

@app.get('/forbidden')
def forbidden():
    abort(403)

with app.test_client() as c:
    print(c.get('/forbidden').status_code)

---

## 9. Sessions et cookies

In [ ]:
app.secret_key = 'dev-secret-change-me'

@app.get('/login/<name>')
def login(name: str):
    session['user'] = name
    return {'logged': name}

@app.get('/whoami')
def whoami():
    return {'user': session.get('user')}

with app.test_client() as c:
    c.get('/login/Alice')
    print(c.get('/whoami').get_json())

In [ ]:
# Cookies directs
from flask import make_response

@app.get('/setcookie')
def setcookie():
    r = make_response({'ok': True})
    r.set_cookie('theme', 'dark', max_age=3600)
    return r

with app.test_client() as c:
    r = c.get('/setcookie')
    print(r.headers.get('Set-Cookie'))

---

## 10. Middlewares et hooks

In [ ]:
@app.before_request
def log_request():
    pass  # print('Requete :', request.method, request.path)

@app.after_request
def add_header(response):
    response.headers['X-App'] = 'demo'
    return response

with app.test_client() as c:
    r = c.get('/')
    print('X-App :', r.headers.get('X-App'))

---

## 11. Configuration

In [ ]:
app3 = Flask(__name__)
app3.config['DEBUG'] = False
app3.config['DATABASE_URI'] = 'sqlite:///app.db'
app3.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024
app3.config

### Strategies de configuration

- `app.config.from_envvar('APP_SETTINGS')`
- `app.config.from_pyfile('config.py')`
- `app.config.from_object('config.Production')`
- Variables d'environnement avec `os.environ.get`

---

## 12. Tests avec le client de test

In [ ]:
import pytest  # noqa: F401

def test_index():
    with app.test_client() as c:
        r = c.get('/')
        assert r.status_code == 200
        assert b'Hello' in r.data

test_index()
print('OK')

In [ ]:
def test_create_note():
    with app.test_client() as c:
        r = c.post('/api/notes', json={'title': 'T'})
        assert r.status_code == 201
        assert r.get_json()['title'] == 'T'

test_create_note()
print('OK')

---

## Synthese

| Element | Flask |
|---|---|
| App | `Flask(__name__)` |
| Route | `@app.route('/path')` ou `@app.get/post/put/delete` |
| JSON | `jsonify(...)` / `request.get_json()` |
| Form | `request.form.get(...)` |
| Template | `render_template_string` / `render_template` |
| Blueprint | `Blueprint('name', __name__, url_prefix='/pref')` |
| Session | `session['key'] = val` |
| Cookie | `make_response` + `set_cookie` |
| Erreurs | `@app.errorhandler(404)` |
| Hooks | `before_request`, `after_request` |
| Tests | `app.test_client()` |

### Regles a retenir

- **Flask = microframework** : on ajoute uniquement ce qu'on utilise.
- **Blueprints** des que l'app grossit.
- **`test_client`** est le moyen standard de tester.
- **`secret_key` obligatoire** pour sessions et flash messages.
- **Pas de rendu HTML en API** : utiliser `jsonify`.

---

## Exercices

### Exercice 1 (facile) -- Echo

Creez une route `/echo` qui renvoie le body JSON recu.

In [ ]:
from flask import Flask, request, jsonify

# TODO

<details>
<summary>Voir la correction</summary>

```python
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.post('/echo')
def echo():
    return jsonify(request.get_json())

with app.test_client() as c:
    print(c.post('/echo', json={'a': 1}).get_json())
```

</details>

### Exercice 2 (facile) -- Param en query string

Route `/search` qui renvoie le parametre `q` recu en query string.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.get('/search')
def search():
    q = request.args.get('q', '')
    return jsonify({'query': q})

with app.test_client() as c:
    print(c.get('/search?q=python').get_json())
```

</details>

### Exercice 3 (moyen) -- CRUD en memoire

Creez une API REST pour gerer des articles : GET /articles, POST /articles, GET /articles/<id>, DELETE /articles/<id>. Stockage en dict.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from flask import Flask, request, jsonify
app = Flask(__name__)
articles = {}
nid = [1]

@app.get('/articles')
def list_articles():
    return jsonify(list(articles.values()))

@app.post('/articles')
def create_article():
    data = request.get_json()
    a = {'id': nid[0], 'title': data['title']}
    articles[nid[0]] = a
    nid[0] += 1
    return jsonify(a), 201

@app.get('/articles/<int:aid>')
def get_article(aid: int):
    if aid not in articles:
        return {'error': 'not found'}, 404
    return articles[aid]

@app.delete('/articles/<int:aid>')
def delete_article(aid: int):
    articles.pop(aid, None)
    return '', 204
```

</details>

### Exercice 4 (moyen) -- Authentification basique par token

Creez un middleware qui verifie la presence d'un header `X-Token: secret` sur toutes les routes commencant par `/admin/`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.before_request
def check_token():
    if request.path.startswith('/admin/'):
        if request.headers.get('X-Token') != 'secret':
            return jsonify({'error': 'unauthorized'}), 401

@app.get('/admin/data')
def admin_data():
    return {'secret': 42}

with app.test_client() as c:
    print(c.get('/admin/data').status_code)
    print(c.get('/admin/data', headers={'X-Token': 'secret'}).get_json())
```

</details>

### Exercice 5 (difficile) -- Blueprint avec pagination

Creez un blueprint `api_v1` qui expose `/api/v1/items` avec pagination (`?page=1&per_page=10`) sur une liste de 100 elements.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from flask import Flask, Blueprint, request, jsonify
api = Blueprint('api_v1', __name__, url_prefix='/api/v1')
ITEMS = [{'id': i, 'name': f'item {i}'} for i in range(100)]

@api.get('/items')
def list_items():
    page = int(request.args.get('page', 1))
    per = int(request.args.get('per_page', 10))
    start = (page - 1) * per
    return jsonify({
        'page': page,
        'per_page': per,
        'total': len(ITEMS),
        'items': ITEMS[start:start + per],
    })

app = Flask(__name__)
app.register_blueprint(api)
with app.test_client() as c:
    print(c.get('/api/v1/items?page=2&per_page=5').get_json())
```

</details>

---

## Ressources externes

- [Documentation Flask](https://flask.palletsprojects.com/)
- [Flask Mega-Tutorial](https://blog.miguelgrinberg.com/post/the-flask-mega-tutorial-part-i-hello-world)
- [Jinja2 documentation](https://jinja.palletsprojects.com/)